# 1. Data and cutoff-safe feature construction

Upload the regenerated cutoff-safe daily series. Section 1 loads the data, defines feature engineering, and displays dataset descriptions. The targets are direct future levels, y(t+h): Avg_Elapsed_Time and Avg_Remaining_Time. After Section 1, each model group in Sections 2–4 runs independently. Its final rolling-origin cell downloads a self-contained ZIP. You can close Colab after downloading it; Section 5 later combines uploaded result ZIPs without retraining.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# --- Source notebook cell boundary ---

from google.colab import files
uploaded = files.upload()
csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_candidates) != 1:
    raise ValueError("Upload exactly one regenerated cutoff-safe daily time-series CSV.")
csv_path = csv_candidates[0]
if "cutoff_safe" not in csv_path.lower():
    raise ValueError("Use the regenerated *_final_online_daily_time_series_cutoff_safe.csv file.")
df = pd.read_csv(csv_path)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).set_index("date")
else:
    df.index = pd.to_datetime(df.index, errors="coerce")
df.index = pd.DatetimeIndex(df.index).normalize()
df = df.loc[~df.index.isna()].sort_index()
if df.index.has_duplicates:
    raise ValueError("Daily dataset must contain one row per date.")

PROCESS_STATE_VARS = [
    "Cases_With_Activity", "Events_During_Day",
    "Transitions_Completed_During_Day", "Active_Resources_During_Day",
]
missing_process = [c for c in PROCESS_STATE_VARS if c not in df.columns]
if missing_process:
    raise ValueError(f"Regenerate the cutoff-safe time series with workload features; missing: {missing_process}")


# --- Source notebook cell boundary ---

# Sort by time if not already sorted
df = df.sort_index()

# 🧪 Hold out the last 20% as final test set
n_test = int(len(df) * 0.20)
df_test_final = df.tail(n_test).copy()
df_train_full = df.drop(df_test_final.index).copy()

df_test_final

# --- Source notebook cell boundary ---

#feature_engineering.py
import pandas as pd
import numpy as np

FEATURE_GROUPS = {}

def feature_engineering(
    df,
    target_var,
    horizon=1,
    target_lags=range(1, 8),
    actor_lags=range(1, 8),
    rolling_windows=(3, 7, 14),
    smooth_window=3
):
    """Build direct future-level labels and cutoff-available predictor groups."""
    global FEATURE_GROUPS
    df_model = df.copy()

    # Direct target: forecast the future day's observed KPI level y[t+h].
    # Avg_Remaining_Time is never included among predictors.
    if target_var not in {"Avg_Elapsed_Time", "Avg_Remaining_Time"}:
        raise ValueError(f"Unsupported target_var: {target_var}")
    df_model["target"] = pd.to_numeric(df_model[target_var], errors="coerce").shift(-horizon)

    performance_features = []
    workload_features = []
    behavior_proportion_features = []
    behavior_count_features = []
    behavior_duration_features = []

    def add_history_features(series_name, family, include_current=False, lags=target_lags):
        cols = []
        if series_name not in df_model.columns:
            return cols
        if include_current:
            cols.append(series_name)
        for lag in lags:
            name = f"{series_name}_lag{lag}"
            df_model[name] = df_model[series_name].shift(lag)
            cols.append(name)
        for window in rolling_windows:
            for stat in ("mean", "std", "max"):
                name = f"{series_name}_rolling_{stat}{window}"
                roll = df_model[series_name].rolling(window)
                df_model[name] = getattr(roll, stat)().shift(1)
                cols.append(name)
        mean7 = f"{series_name}_rolling_mean7"
        std7 = f"{series_name}_rolling_std7"
        if mean7 in df_model.columns and std7 in df_model.columns:
            zname = f"{series_name}_zscore7"
            df_model[zname] = (df_model[series_name].shift(1) - df_model[mean7]) / df_model[std7]
            cols.append(zname)
        return cols

    # Variant 1: history of average elapsed time only (for both target outcomes).
    performance_features += add_history_features("Avg_Elapsed_Time", "performance", include_current=False)

    # Variant 2 additions: end-of-day activity, event, transition, and resource volume.
    for var in PROCESS_STATE_VARS:
        df_model[var] = pd.to_numeric(df_model[var], errors="coerce")
        workload_features += add_history_features(var, "workload", include_current=True)

    # Discover behavior classes from this dataset's own columns (e.g. HD in BPIC2017).
    count_vars = sorted(c for c in df_model.columns if c.startswith("Cum_Count_"))
    duration_vars = sorted(c for c in df_model.columns if c.startswith("Cum_Time_") and c.endswith("_seconds"))
    count_vars = sorted(c for c in df_model.columns if c.startswith("Cum_Count_"))
    duration_vars = sorted(
        c for c in df_model.columns
        if c.startswith("Cum_Time_") and c.endswith("_seconds")
    )

    if count_vars:
        count_matrix = pd.DataFrame(
            {
                col: pd.to_numeric(df_model[col], errors="coerce").to_numpy()
                for col in count_vars
            },
            index=df_model.index
        )
    else:
        count_matrix = pd.DataFrame(index=df_model.index)

    total_count = (
        count_matrix.sum(axis=1)
        if count_vars
        else pd.Series(0.0, index=df_model.index)
    )

    # Normalized behavior mix plus separate raw-count and duration groups.
    for var in count_vars:
        class_name = var[len("Cum_Count_"):]
        prop_name = f"Behavior_Proportion_{class_name}"
        df_model[prop_name] = count_matrix[var].div(total_count.replace(0, np.nan)).fillna(0.0)
        behavior_proportion_features += add_history_features(prop_name, "proportion", include_current=True, lags=actor_lags)
        behavior_count_features += add_history_features(var, "count", include_current=True, lags=actor_lags)
    for var in duration_vars:
        df_model[var] = pd.to_numeric(df_model[var], errors="coerce")
        behavior_duration_features += add_history_features(var, "duration", include_current=True, lags=actor_lags)

    # Full actor variant = workload-aware baseline plus all behavior families.
    actor_behavior_features = behavior_proportion_features + behavior_count_features + behavior_duration_features
    all_nonperformance_features = workload_features + actor_behavior_features
    df_model.dropna(subset=["target"], inplace=True)

    performance_features = list(dict.fromkeys(performance_features))
    workload_features = list(dict.fromkeys(workload_features))
    behavior_proportion_features = list(dict.fromkeys(behavior_proportion_features))
    behavior_count_features = list(dict.fromkeys(behavior_count_features))
    behavior_duration_features = list(dict.fromkeys(behavior_duration_features))
    all_nonperformance_features = list(dict.fromkeys(all_nonperformance_features))
    FEATURE_GROUPS = {
        "performance": performance_features,
        "workload": workload_features,
        "proportions": behavior_proportion_features,
        "counts": behavior_count_features,
        "durations": behavior_duration_features,
        "actor_behavior": actor_behavior_features,
    }
    return df_model.copy(), performance_features, all_nonperformance_features



# Shared output folder for reusable training and evaluation artifacts.
from pathlib import Path
import json, joblib, zipfile, shutil
ARTIFACT_ROOT = Path("/content/direct_prediction_artifacts")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "models").mkdir(exist_ok=True)
(ARTIFACT_ROOT / "predictions").mkdir(exist_ok=True)
(ARTIFACT_ROOT / "design_matrices").mkdir(exist_ok=True)
(ARTIFACT_ROOT / "metrics").mkdir(exist_ok=True)
raw_input = df.copy()
raw_input.to_csv(ARTIFACT_ROOT / "input_daily_timeseries.csv", index=True, index_label="date")
pd.DataFrame({"date": df.index, "partition": ["train" if i < len(df_train_full) else "holdout" for i in range(len(df))]}).to_csv(ARTIFACT_ROOT / "chronological_split.csv", index=False)
run_config = {"targets": ["Avg_Elapsed_Time", "Avg_Remaining_Time"], "horizons_days": [1, 3, 7], "target_mode": "direct future KPI level y[t+h]", "evaluation": "expanding window; train/test origins 70/10, 80/10, 90/10; three non-overlapping test windows", "source_notebook": "ExtensionMLP4PM", "input_csv": csv_path, "feature_variants": ["Performance-only", "Workload-aware", "Workload+Actor"]}
(ARTIFACT_ROOT / "run_config.json").write_text(json.dumps(run_config, indent=2))


def run_primary_rolling_evaluation(ROLL_GROUP):
    """Train/evaluate one model family on the fixed 70/10, 80/10, 90/10 folds and download its bundle."""
    if ROLL_GROUP not in {"classical","tree","deep"}:
        raise ValueError("ROLL_GROUP must be classical, tree, or deep")
    import os, json, shutil, joblib
    from pathlib import Path
    from tqdm.auto import tqdm
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    if ROLL_GROUP == "tree":
        from xgboost import XGBRegressor
        from lightgbm import LGBMRegressor
    ROLL_ROOT=ARTIFACT_ROOT/"rolling_origin_evaluation"/ROLL_GROUP
    for _sub in ["metrics","predictions","models","design_matrices","sequences","shap"]: (ROLL_ROOT/_sub).mkdir(parents=True,exist_ok=True)
    ROLL_SPECS=[("fold_70_10",0.70,0.80),("fold_80_10",0.80,0.90),("fold_90_10",0.90,1.00)]
    ROLL_TARGETS=["Avg_Elapsed_Time","Avg_Remaining_Time"]
    ROLL_HORIZONS=[1,3,7]
    ROLL_VARIANTS={"PerformanceOnly":"baseline","WorkloadAware":"workload","WorkloadPlusActor":"actor"}
    roll_metric_rows=[]

    def roll_mase(y_true,y_pred,train_series):
        scale=float(np.mean(np.abs(np.diff(np.asarray(train_series,dtype=float))))) if len(train_series)>1 else np.nan
        return float(mean_absolute_error(y_true,y_pred)/scale) if np.isfinite(scale) and scale>0 else np.nan

    def roll_scores(y_true,y_pred,train_series):
        y_true=np.asarray(y_true,dtype=float); y_pred=np.asarray(y_pred,dtype=float)
        return {"RMSE":float(np.sqrt(mean_squared_error(y_true,y_pred))),"MAE":float(mean_absolute_error(y_true,y_pred)),
                "R2":float(r2_score(y_true,y_pred)) if len(y_true)>1 else np.nan,"MASE":roll_mase(y_true,y_pred,train_series)}

    def roll_regression_feature_screen(X_train,y_train,X_test,max_features=25):
        """Match the classical screen while retaining reusable preprocessing objects."""
        from sklearn.impute import SimpleImputer
        from sklearn.preprocessing import StandardScaler
        from sklearn.feature_selection import f_regression
        X_train=X_train.replace([np.inf,-np.inf],np.nan).copy(); X_test=X_test.replace([np.inf,-np.inf],np.nan).copy()
        cols=[c for c in X_train.columns if X_train[c].notna().sum()>0 and X_train[c].nunique(dropna=True)>1]
        if not cols: raise ValueError("No usable classical predictors after filtering.")
        X_train=X_train[cols]; X_test=X_test[cols]
        imputer=SimpleImputer(strategy="median"); scaler=StandardScaler()
        Xtr_i=imputer.fit_transform(X_train); Xte_i=imputer.transform(X_test)
        Xtr=pd.DataFrame(scaler.fit_transform(Xtr_i),columns=cols,index=X_train.index)
        Xte=pd.DataFrame(scaler.transform(Xte_i),columns=cols,index=X_test.index)
        scores,_=f_regression(Xtr,np.asarray(y_train)); scores=np.nan_to_num(scores,nan=-np.inf,posinf=-np.inf,neginf=-np.inf)
        selected=list(Xtr.columns[np.argsort(scores)[::-1][:min(max_features,Xtr.shape[1])]])
        return Xtr[selected],Xte[selected],selected,imputer,scaler,cols

    def roll_save(model_group,model_name,fold,target,horizon,variant,origins,y_true,y_pred,n_train,train_values,model=None,features=None,design=None,sequence=None):
        safe=f"{model_group}_{model_name}_{fold}_{target}_h{horizon}_{variant}".replace("+","_").replace("/","_")
        origins=pd.Index(origins); target_dates=[]
        for date in origins: target_dates.append(df.index[df.index.get_loc(date)+horizon])
        scores=roll_scores(y_true,y_pred,train_values)
        roll_metric_rows.append({"group":model_group,"model":model_name,"fold":fold,"target":target,"horizon_days":horizon,
            "variant":variant,"n_train":int(n_train),"n_test":int(len(y_true)),"n_train_raw":int(train_end_global),
            "n_test_origins_before_horizon_purge":int(test_raw_global),**scores})
        pd.DataFrame({"fold":fold,"cutoff_date":origins,"target_date":target_dates,"y_true":y_true,"y_pred":y_pred,
            "error":np.asarray(y_true)-np.asarray(y_pred)}).to_csv(ROLL_ROOT/"predictions"/f"{safe}.csv",index=False)
        if model is not None: joblib.dump(model,ROLL_ROOT/"models"/f"{safe}.joblib")
        if design is not None:
            design.assign(target=np.asarray(y_true),prediction=np.asarray(y_pred)).to_csv(ROLL_ROOT/"design_matrices"/f"{safe}.csv",index_label="cutoff_date")
        if sequence is not None:
            np.savez_compressed(ROLL_ROOT/"sequences"/f"{safe}.npz",X=sequence.get("X_test_scaled",sequence.get("X_test")),
                feature_names=np.asarray(sequence.get("feature_names",[])),dates=np.asarray(origins.astype(str)),
                y_true=np.asarray(y_true),y_pred=np.asarray(y_pred))

    for fold_name,train_frac,test_end_frac in ROLL_SPECS:
      n=len(df); train_end=int(np.floor(n*train_frac)); test_end=int(np.floor(n*test_end_frac))
      for target_var in ROLL_TARGETS:
        for horizon in ROLL_HORIZONS:
          train_raw=df.iloc[:train_end].copy()
          # Purge trailing test origins whose y[t+h] would fall outside this window.
          test_raw=df.iloc[train_end:test_end].copy()
          if len(test_raw)==0: print(f"Skip {fold_name}, {target_var}, h={horizon}: empty test block after purge"); continue
          train_end_global=len(train_raw); test_raw_global=len(test_raw)
          train_fe,performance_features,_=feature_engineering(train_raw,target_var=target_var,horizon=horizon)
          groups=dict(FEATURE_GROUPS)
          context=pd.concat([train_raw.tail(200),test_raw])
          test_fe,_,_=feature_engineering(context,target_var=target_var,horizon=horizon)
          test_fe = test_fe.loc[test_fe.index.isin(test_raw.index)] #changed for 2012
          train_values=pd.to_numeric(train_raw[target_var],errors="coerce").dropna().to_numpy(dtype=float)
          p_features=groups["performance"]
          w_features=p_features+groups["workload"]
          a_features=w_features+groups["proportions"]+groups["counts"]+groups["durations"]
          variant_features={"PerformanceOnly":p_features,"WorkloadAware":w_features,"WorkloadPlusActor":a_features,
              "ProportionsOnly":w_features+groups["proportions"],"CountsOnly":w_features+groups["counts"],"DurationsOnly":w_features+groups["durations"]}
          for variant,features in variant_features.items():
            if not features:
              print(f"Skip {fold_name} {target_var} h{horizon} {variant}: no features")
              continue

            tr = train_fe.dropna(subset=features + ["target"]).copy()
            te = test_fe.dropna(subset=features + ["target"]).copy()

            if len(tr) < 10 or len(te) == 0:
              print(
                  f"Skip {fold_name} {target_var} h{horizon} {variant}: "
                  f"usable train rows={len(tr)}, test rows={len(te)}"
              )
              continue
            tr=train_fe.dropna(subset=features+["target"]).copy(); te=test_fe.dropna(subset=features+["target"]).copy()
            if len(tr)<10 or len(te)==0: continue
            Xtr=tr[features]; Xte=te[features]; ytr=tr["target"].astype(float).to_numpy(); yte=te["target"].astype(float).to_numpy()
            # Classical methods and primary XGBoost/LightGBM variants plus XGBoost ablations.
            if ROLL_GROUP=="classical" and variant in ROLL_VARIANTS:
              for model_name in ["SARIMAX","ARX-HAC"]:
                try:
                  Xa,Xb,selected,screen_imputer,screen_scaler,screen_input_columns=roll_regression_feature_screen(Xtr,ytr,Xte,max_features=25)
                  if model_name=="SARIMAX": fitted,pred=fit_predict_sarimax(Xa,pd.Series(ytr),Xb)
                  else: fitted,pred=fit_predict_arx_hac(Xa,pd.Series(ytr),Xb,hac_lags=max(1,horizon))
                  roll_save("Classical",model_name,fold_name,target_var,horizon,variant,te.index,yte,np.asarray(pred),len(tr),train_values,
                      model={"fitted_model":fitted,"selected_features":selected,"input_columns":screen_input_columns,
                          "imputer":screen_imputer,"scaler":screen_scaler,"target_mode":"direct"},features=selected,design=Xb)
                except Exception as e: print(f"Classical skip {model_name} {fold_name} {target_var} h{horizon} {variant}: {e}")
            if ROLL_GROUP=="tree":
              for model_name,params in [("XGBoost",best_xgb_params),("LightGBM",best_lgbm_params)]:
                if variant not in ROLL_VARIANTS:
                  if model_name!="XGBoost": continue
                try:
                  model=XGBRegressor(**params) if model_name=="XGBoost" else LGBMRegressor(**params)
                  model.fit(Xtr,ytr); pred=model.predict(Xte)
                  roll_save("Tree",model_name,fold_name,target_var,horizon,variant,te.index,yte,pred,len(tr),train_values,
                      model={"model":model,"features":list(features),"params":params,"target_mode":"direct"},design=Xte)
                  if model_name=="XGBoost" and variant=="WorkloadPlusActor":
                    try:
                      import shap
                      sv=np.asarray(shap.TreeExplainer(model).shap_values(Xte))
                      np.savez_compressed(ROLL_ROOT/"shap"/f"XGBoost_{fold_name}_{target_var}_h{horizon}_{variant}.npz",
                          values=sv,feature_names=np.asarray(features),feature_values=Xte.to_numpy(),dates=np.asarray(te.index.astype(str)))
                    except Exception as e: print("SHAP skipped:",e)
                except Exception as e: print(f"Tree skip {model_name} {fold_name} {target_var} h{horizon} {variant}: {e}")
          # Recurrent architectures use the already-selected training settings.
          for variant,feature_set in (ROLL_VARIANTS.items() if ROLL_GROUP=="deep" else []):
            for rnn_type in ["lstm","gru"]:
              for attention in [False,True]:
                try:
                  fn=final_evaluate_rnn_with_attention if attention else final_evaluate_rnn_for_target_horizon
                  result=fn(train_raw,test_raw,target_var=target_var,horizon=horizon,feature_set=feature_set,rnn_type=rnn_type,
                      time_steps=TIME_STEPS,seed=42)
                  model_name=f"{rnn_type.upper()}+{'Attention' if attention else 'NoAttention'}"
                  # Copy each fitted model before the next fold/variant overwrites its helper path.
                  model_path=result.get("model_path")
                  saved_model=None
                  if model_path and Path(model_path).exists():
                    dest=ROLL_ROOT/"models"/f"RNN_{model_name}_{fold_name}_{target_var}_h{horizon}_{variant}.keras"
                    shutil.copy2(model_path,dest); saved_model=str(dest)
                  elif result.get("model") is not None:
                    dest=ROLL_ROOT/"models"/f"RNN_{model_name}_{fold_name}_{target_var}_h{horizon}_{variant}.keras"
                    result["model"].save(dest); saved_model=str(dest)
                  prep_path=ROLL_ROOT/"models"/f"RNN_{model_name}_{fold_name}_{target_var}_h{horizon}_{variant}.preprocessing.joblib"
                  joblib.dump({"scaler_x":result.get("scaler_x"),"scaler_y":result.get("scaler_y"),
                      "features":result.get("feature_names",result.get("features",[])),"time_steps":TIME_STEPS,
                      "target":target_var,"horizon_days":horizon,"variant":variant,"fold":fold_name},prep_path)
                  roll_save("DeepLearning",model_name,fold_name,target_var,horizon,variant,result["index"],result["y_true"],result["y_pred"],
                      int(result.get("n_train_sequences",len(train_raw))),train_values,model={"model_path":saved_model,"feature_set":feature_set,"target_mode":"direct"},sequence=result)
                except Exception as e: print(f"RNN skip {rnn_type} attention={attention} {fold_name} {target_var} h{horizon} {variant}: {e}")

    fold_metrics=pd.DataFrame(roll_metric_rows)
    fold_metrics.to_csv(ROLL_ROOT/"metrics"/"rolling_origin_fold_metrics.csv",index=False)
    if len(fold_metrics):
      summary_rows=[]
      keycols=["group","model","target","horizon_days","variant"]
      for keys,g in fold_metrics.groupby(keycols,dropna=False):
        row=dict(zip(keycols,keys)); row["n_folds"]=int(g["fold"].nunique())
        for metric in ["RMSE","MAE","R2","MASE"]:
          vals=pd.to_numeric(g[metric],errors="coerce").dropna()
          row[f"{metric}_fold_scores"]="; ".join(f"{fold}:{value:.6g}" for fold,value in zip(g.loc[vals.index,"fold"],vals))
          row[f"{metric}_mean"]=float(vals.mean()) if len(vals) else np.nan
          row[f"{metric}_sd"]=float(vals.std(ddof=1)) if len(vals)>1 else np.nan
          row[f"{metric}_min"]=float(vals.min()) if len(vals) else np.nan
          row[f"{metric}_max"]=float(vals.max()) if len(vals) else np.nan
          row[f"{metric}_range"]=float(vals.max()-vals.min()) if len(vals) else np.nan
        summary_rows.append(row)
      summary_df=pd.DataFrame(summary_rows)
      summary_df.to_csv(ROLL_ROOT/"metrics"/"rolling_origin_across_fold_summary.csv",index=False)
      display(fold_metrics.head(20)); display(summary_df.head(20))
    else: summary_df=pd.DataFrame(); print("No rolling-origin metrics were produced; inspect skip/error messages.")

    fold_counts=(fold_metrics.groupby(["fold","group","model","target","horizon_days","variant"],as_index=False)
                 .agg(n_train_min=("n_train","min"),n_test_min=("n_test","min"),n_test_max=("n_test","max"))) if len(fold_metrics) else pd.DataFrame()
    fold_counts.to_csv(ROLL_ROOT/"metrics"/"effective_sample_counts_by_fold.csv",index=False)
    roll_config={"folds":ROLL_SPECS,"targets":ROLL_TARGETS,"horizons":ROLL_HORIZONS,"variants":list(ROLL_VARIANTS),
      "direct_targets":True,"test_windows_non_overlapping":True,"trailing_horizon_origins_purged_from_each_test_window":True,
      "mase_scaling":"fold-specific mean absolute first difference of training target series",
      "settings":"model settings fixed from prior validation/tuning; no retuning in these final origins","model_group":ROLL_GROUP}
    (ROLL_ROOT/"run_config.json").write_text(json.dumps(roll_config,indent=2))
    # Include the manuscript dataset summaries and exact input series in the final bundle.
    DATASET_EXPORT=ROLL_ROOT/"dataset_metadata"; DATASET_EXPORT.mkdir(parents=True,exist_ok=True)
    for _name in ["dataset_characteristics.csv","behavior_type_distribution.csv","input_daily_timeseries.csv","chronological_split.csv","run_config.json"]:
      _source=ARTIFACT_ROOT/_name
      if _source.exists(): shutil.copy2(_source,DATASET_EXPORT/_name)
    roll_zip=shutil.make_archive(str(ARTIFACT_ROOT/f"{ROLL_GROUP}_rolling_origin_results"),"zip",root_dir=ROLL_ROOT)
    print("Rolling-origin results, predictions, models, design matrices, counts, SHAP values, and summaries saved at:",ROLL_ROOT)
    from google.colab import files
    files.download(roll_zip)


Saving BPIC2011_final_online_daily_time_series_cutoff_safe_days.csv to BPIC2011_final_online_daily_time_series_cutoff_safe_days.csv


## 1.1 Dataset characteristics for the manuscript

Run this after loading the daily CSV. When prompted, upload the matching raw event log (`.xes`, `.xes.gz`, or a ZIP containing one XES file) and the exact enriched actor-behavior CSV used to generate the daily series. This reports the calendar span, daily observations, resource coverage, and behavior-class distribution. Effective training and test counts after feature and horizon filtering are saved by the rolling-origin run cell in each model section.


In [ ]:
# DATASET CHARACTERISTICS: raw log + behavior table
import os, sys, zipfile, subprocess, importlib.util
from pathlib import Path
from google.colab import files

if importlib.util.find_spec("pm4py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pm4py"])
import pm4py

print("Upload the raw event log (XES, XES.GZ, or ZIP).")
raw_upload = files.upload()
raw_names = list(raw_upload)
meta_logs = [n for n in raw_names if n.lower().endswith((".xes", ".xes.gz", ".zip"))]
if len(meta_logs) != 1:
    raise ValueError("Upload exactly one raw XES, XES.GZ, or ZIP file in this step.")
meta_log_path = meta_logs[0]

print("Upload the enriched actor-behavior CSV.")
behavior_upload = files.upload()
behavior_names = list(behavior_upload)
meta_csvs = [n for n in behavior_names if n.lower().endswith(".csv")]
if len(meta_csvs) != 1:
    raise ValueError("Upload exactly one enriched actor-behavior CSV in this step.")

if meta_log_path.lower().endswith(".zip"):
    with zipfile.ZipFile(meta_log_path) as zf:
        xes_names = [
            n for n in zf.namelist()
            if n.lower().endswith((".xes", ".xes.gz"))
        ]
        if len(xes_names) != 1:
            raise ValueError("The ZIP must contain exactly one XES or XES.GZ file.")
        Path("/content/metadata_raw_log").mkdir(parents=True, exist_ok=True)
        meta_log_path = zf.extract(xes_names[0], "/content/metadata_raw_log")

log_meta = pm4py.read_xes(meta_log_path)
events_meta = pm4py.convert_to_dataframe(log_meta)
actor_meta = pd.read_csv(meta_csvs[0], low_memory=False)

resource_candidates = [
    c for c in ["org:resource", "org:group"]
    if c in events_meta.columns
]
resource_col_meta = resource_candidates[0] if resource_candidates else None
missing_resource_fraction = (
    float(events_meta[resource_col_meta].isna().mean())
    if resource_col_meta else np.nan
)
resource_count = (
    int(events_meta[resource_col_meta].nunique(dropna=True))
    if resource_col_meta else np.nan
)

behavior_col = "behavior" if "behavior" in actor_meta.columns else None
behavior_distribution = (
    actor_meta[behavior_col].fillna("<missing>").value_counts(dropna=False)
    .rename_axis("behavior").reset_index(name="transition_count")
    if behavior_col
    else pd.DataFrame(columns=["behavior", "transition_count"])
)

dataset_characteristics = pd.DataFrame([{
    "dataset_file": Path(csv_path).name,
    "calendar_start": df.index.min().date(),
    "calendar_end": df.index.max().date(),
    "daily_observations": int(len(df)),
    "raw_event_rows": int(len(events_meta)),
    "resource_column": resource_col_meta,
    "distinct_resources_or_groups": resource_count,
    "missing_resource_fraction": missing_resource_fraction,
    "behavior_table_rows": int(len(actor_meta)),
    "behavior_labels": (
        ", ".join(sorted(actor_meta[behavior_col].dropna().astype(str).unique()))
        if behavior_col else "unavailable"
    )
}])

display(dataset_characteristics)
display(behavior_distribution)
dataset_characteristics.to_csv(
    ARTIFACT_ROOT / "dataset_characteristics.csv", index=False
)
behavior_distribution.to_csv(
    ARTIFACT_ROOT / "behavior_type_distribution.csv", index=False
)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Upload the raw event log (XES, XES.GZ, or ZIP).


Saving Hospital_log.xes.gz to Hospital_log.xes.gz
Upload the enriched actor-behavior CSV.


Saving BPIC2011_actor_behavior_results.csv to BPIC2011_actor_behavior_results.csv


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1143 [00:00<?, ?it/s]

,dataset_file,calendar_start,calendar_end,daily_observations,raw_event_rows,resource_column,distinct_resources_or_groups,missing_resource_fraction,behavior_table_rows,behavior_labels
0,BPIC2011_final_online_daily_time_series_cutoff...,2005-01-03,2008-03-20,1173,150291,org:group,42,0.000106,149132,"C, HB, HI, I"


,behavior,transition_count
0,C,116606
1,HI,23653
2,HB,5916
3,I,2957


# 2. Classical time-series models

Run after Section 1. The helper cell defines SARIMAX and ARX-HAC. The next code cell trains both models on the three expanding-window blocks and downloads reusable results.


In [ ]:
# CLASSICAL MODEL HELPERS FOR ROLLING-ORIGIN EVALUATION
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
SARIMAX_ORDER=(1,0,0)
SARIMAX_SEASONAL_ORDER=(1,0,0,7)
def fit_predict_sarimax(X_train, y_train, X_test):
    Xtr, Xte = np.asarray(X_train, float), np.asarray(X_test, float)
    ytr = np.asarray(y_train, float).ravel()
    fit = SARIMAX(ytr, exog=Xtr, order=SARIMAX_ORDER,
                  seasonal_order=SARIMAX_SEASONAL_ORDER, trend="c",
                  enforce_stationarity=False, enforce_invertibility=False
                  ).fit(disp=False, maxiter=300)
    return fit, pd.Series(np.asarray(fit.forecast(len(Xte), exog=Xte)).ravel())

def fit_predict_arx_hac(X_train, y_train, X_test, hac_lags=7):
    Xtr = sm.add_constant(np.asarray(X_train, float), has_constant="add")
    Xte = sm.add_constant(np.asarray(X_test, float), has_constant="add")
    ytr = np.asarray(y_train, float).ravel()
    fit = sm.OLS(ytr, Xtr).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    return fit, pd.Series(np.asarray(fit.predict(Xte)).ravel())

The next cell trains both classical models on the three expanding-window blocks and downloads their rolling-origin results bundle.


In [ ]:
run_primary_rolling_evaluation("classical")


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/tmp/ipykernel_1084/284874165.py:21: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(ytr, Xtr).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/tmp/ipykernel_1084/284874165.py:21: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(ytr, Xtr).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
/tmp/ipykernel_1084/284874165.py:21: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = s

,group,model,fold,target,horizon_days,variant,n_train,n_test,n_train_raw,n_test_origins_before_horizon_purge,RMSE,MAE,R2,MASE
0,Classical,SARIMAX,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,799,116,821,117,105.951132,84.305725,-0.333437,2.507780
1,Classical,ARX-HAC,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,799,116,821,117,96.517107,76.360911,-0.106547,2.271451
2,Classical,SARIMAX,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,799,116,821,117,103.757543,81.976781,-0.278794,2.438502
3,Classical,ARX-HAC,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,799,116,821,117,96.574222,75.689520,-0.107857,2.251480
4,Classical,SARIMAX,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,797,116,821,117,101.687844,81.701255,-0.228285,2.430306
5,Classical,ARX-HAC,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,797,116,821,117,93.666560,74.780293,-0.042150,2.224434
6,Classical,SARIMAX,fold_70_10,Avg_Elapsed_Time,3,PerformanceOnly,797,114,821,117,101.812422,81.666960,-0.278984,2.429286
7,Classical,ARX-HAC,fold_70_10,Avg_Elapsed_Time,3,PerformanceOnly,797,114,821,117,93.578393,75.121848,-0.080475,2.234594
8,Classical,SARIMAX,fold_70_10,Avg_Elapsed_Time,3,WorkloadAware,797,114,821,117,98.105757,77.685674,-0.187551,2.310858
9,Classical,ARX-HAC,fold_70_10,Avg_Elapsed_Time,3,WorkloadAware,797,114,821,117,91.558772,73.271194,-0.034340,2.179544


,group,model,target,horizon_days,variant,n_folds,RMSE_fold_scores,RMSE_mean,RMSE_sd,RMSE_min,...,R2_sd,R2_min,R2_max,R2_range,MASE_fold_scores,MASE_mean,MASE_sd,MASE_min,MASE_max,MASE_range
0,Classical,ARX-HAC,Avg_Elapsed_Time,1,PerformanceOnly,3,fold_70_10:96.5171; fold_80_10:92.2026; fold_9...,96.388235,4.122698,92.202612,...,0.164927,-0.136978,0.162681,0.299659,fold_70_10:2.27145; fold_80_10:1.94403; fold_9...,2.003300,0.243978,1.794415,2.271451,0.477036
1,Classical,ARX-HAC,Avg_Elapsed_Time,1,WorkloadAware,3,fold_70_10:96.5742; fold_80_10:92.138; fold_90...,96.055275,3.685326,92.137982,...,0.174182,-0.135385,0.179128,0.314513,fold_70_10:2.25148; fold_80_10:1.90599; fold_9...,1.982243,0.240359,1.789262,2.251480,0.462217
2,Classical,ARX-HAC,Avg_Elapsed_Time,1,WorkloadPlusActor,3,fold_70_10:93.6666; fold_80_10:89.2558; fold_9...,95.349456,7.086549,89.255843,...,0.108765,-0.065464,0.133494,0.198958,fold_70_10:2.22443; fold_80_10:1.83387; fold_9...,1.959540,0.229505,1.820313,2.224434,0.404120
3,Classical,ARX-HAC,Avg_Elapsed_Time,3,PerformanceOnly,3,fold_70_10:93.5784; fold_80_10:94.9158; fold_9...,94.085017,0.725225,93.578393,...,0.232517,-0.188477,0.257244,0.445721,fold_70_10:2.23459; fold_80_10:2.00921; fold_9...,1.971239,0.284248,1.669914,2.234594,0.564680
4,Classical,ARX-HAC,Avg_Elapsed_Time,3,WorkloadAware,3,fold_70_10:91.5588; fold_80_10:90.1451; fold_9...,92.385194,2.748103,90.145134,...,0.164695,-0.072009,0.230215,0.302224,fold_70_10:2.17954; fold_80_10:1.86381; fold_9...,1.925322,0.229727,1.732614,2.179544,0.446929
5,Classical,ARX-HAC,Avg_Elapsed_Time,3,WorkloadPlusActor,3,fold_70_10:92.8888; fold_80_10:92.1504; fold_9...,97.202297,8.119057,92.150441,...,0.078028,-0.120234,0.033849,0.154083,fold_70_10:2.20241; fold_80_10:1.93327; fold_9...,2.006705,0.171238,1.884434,2.202413,0.317979
6,Classical,ARX-HAC,Avg_Elapsed_Time,7,PerformanceOnly,3,fold_70_10:91.1291; fold_80_10:92.0333; fold_9...,95.019950,5.973234,91.129102,...,0.134353,-0.104121,0.136621,0.240742,fold_70_10:2.25133; fold_80_10:1.86514; fold_9...,2.000473,0.217478,1.865143,2.251334,0.386191
7,Classical,ARX-HAC,Avg_Elapsed_Time,7,WorkloadAware,3,fold_70_10:87.1311; fold_80_10:85.7403; fold_9...,91.293377,8.442450,85.740312,...,0.075821,0.006180,0.151616,0.145436,fold_70_10:2.15964; fold_80_10:1.72403; fold_9...,1.923115,0.220204,1.724032,2.159640,0.435608
8,Classical,ARX-HAC,Avg_Elapsed_Time,7,WorkloadPlusActor,3,fold_70_10:81.913; fold_80_10:86.3297; fold_90...,90.155352,10.681946,81.913006,...,0.053551,0.028490,0.121650,0.093161,fold_70_10:1.97869; fold_80_10:1.80913; fold_9...,1.868401,0.095599,1.809128,1.978686,0.169558
9,Classical,ARX-HAC,Avg_Remaining_Time,1,PerformanceOnly,3,fold_70_10:70.3885; fold_80_10:71.9242; fold_9...,71.431030,0.903312,70.388482,...,5.970417,-14.672859,-2.849204,11.823655,fold_70_10:0.848511; fold_80_10:1.04066; fold_...,1.001585,0.137758,0.848511,1.115583,0.267073


Rolling-origin results, predictions, models, design matrices, counts, SHAP values, and summaries saved at: /content/direct_prediction_artifacts/rolling_origin_evaluation/classical


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 3. Tree-based models

Run after Section 1. The settings cell defines the fixed XGBoost and LightGBM parameters. The rolling-origin cell trains all three input variants and saves XGBoost SHAP values for WorkloadPlusActor.


In [ ]:
# FIXED TREE-MODEL SETTINGS FOR ROLLING-ORIGIN EVALUATION
best_xgb_params={"n_estimators":1500,"learning_rate":0.05,"max_depth":6,
    "subsample":0.9,"colsample_bytree":0.9,"gamma":0,"min_child_weight":1,"random_state":42}
best_lgbm_params={"n_estimators":1500,"learning_rate":0.05,"max_depth":5,
    "bagging_fraction":0.9,"feature_fraction":0.9,"reg_alpha":0,"reg_lambda":0,
    "min_child_samples":1,"random_state":42}


The rolling-origin evaluator trains XGBoost and LightGBM for all three variants and saves fold predictions, models, design matrices, and XGBoost SHAP values for WorkloadPlusActor.


The next cell trains the tree models on the three expanding-window blocks and downloads their results bundle.


In [ ]:
run_primary_rolling_evaluation("tree")


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

,group,model,fold,target,horizon_days,variant,n_train,n_test,n_train_raw,n_test_origins_before_horizon_purge,RMSE,MAE,R2,MASE
0,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,799,116,821,117,122.821554,97.391048,-0.791886,2.897019
1,Tree,LightGBM,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,799,116,821,117,113.058202,87.745375,-0.518327,2.610096
2,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,799,116,821,117,101.025702,77.819326,-0.212341,2.314833
3,Tree,LightGBM,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,799,116,821,117,98.683920,74.556982,-0.156788,2.217791
4,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,797,116,821,117,96.421201,74.233640,-0.104349,2.208173
5,Tree,LightGBM,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,797,116,821,117,93.849141,73.532499,-0.046217,2.187316
6,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,ProportionsOnly,799,116,821,117,97.127160,75.998298,-0.120579,2.260665
7,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,CountsOnly,797,116,821,117,92.431579,70.765908,-0.014850,2.105021
8,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,1,DurationsOnly,797,116,821,117,95.072880,72.902936,-0.073679,2.168589
9,Tree,XGBoost,fold_70_10,Avg_Elapsed_Time,3,PerformanceOnly,797,114,821,117,125.520762,102.089946,-0.943993,3.036794


,group,model,target,horizon_days,variant,n_folds,RMSE_fold_scores,RMSE_mean,RMSE_sd,RMSE_min,...,R2_sd,R2_min,R2_max,R2_range,MASE_fold_scores,MASE_mean,MASE_sd,MASE_min,MASE_max,MASE_range
0,Tree,LightGBM,Avg_Elapsed_Time,1,PerformanceOnly,3,fold_70_10:113.058; fold_80_10:95.8575; fold_9...,108.899934,11.539587,95.857512,...,0.193413,-0.518327,-0.151350,0.366977,fold_70_10:2.6101; fold_80_10:2.08757; fold_90...,2.276756,0.289558,2.087566,2.610096,0.522531
1,Tree,LightGBM,Avg_Elapsed_Time,1,WorkloadAware,3,fold_70_10:98.6839; fold_80_10:90.2953; fold_9...,100.826973,11.750710,90.295281,...,0.045711,-0.156788,-0.069151,0.087637,fold_70_10:2.21779; fold_80_10:1.94334; fold_9...,2.071843,0.138056,1.943336,2.217791,0.274455
2,Tree,LightGBM,Avg_Elapsed_Time,1,WorkloadPlusActor,3,fold_70_10:93.8491; fold_80_10:106.241; fold_9...,102.403260,7.421120,93.849141,...,0.304755,-0.509549,0.065075,0.574624,fold_70_10:2.18732; fold_80_10:2.21834; fold_9...,2.114750,0.153339,1.938598,2.218336,0.279738
3,Tree,LightGBM,Avg_Elapsed_Time,3,PerformanceOnly,3,fold_70_10:123.256; fold_80_10:96.9672; fold_9...,113.529702,14.416132,96.967192,...,0.370876,-0.874466,-0.224085,0.650381,fold_70_10:3.03331; fold_80_10:2.06032; fold_9...,2.469328,0.504669,2.060319,3.033311,0.972992
4,Tree,LightGBM,Avg_Elapsed_Time,3,WorkloadAware,3,fold_70_10:100.484; fold_80_10:100.448; fold_9...,106.542960,10.525469,100.448432,...,0.070876,-0.331068,-0.190364,0.140704,fold_70_10:2.36628; fold_80_10:2.10279; fold_9...,2.235794,0.131763,2.102794,2.366284,0.263490
5,Tree,LightGBM,Avg_Elapsed_Time,3,WorkloadPlusActor,3,fold_70_10:97.9915; fold_80_10:110.866; fold_9...,107.449769,8.295669,97.991528,...,0.281353,-0.621469,-0.095783,0.525686,fold_70_10:2.32705; fold_80_10:2.25817; fold_9...,2.215284,0.138297,2.060624,2.327055,0.266431
6,Tree,LightGBM,Avg_Elapsed_Time,7,PerformanceOnly,3,fold_70_10:101.286; fold_80_10:95.7583; fold_9...,104.848072,11.299851,95.758345,...,0.101685,-0.342957,-0.148016,0.194941,fold_70_10:2.49378; fold_80_10:1.98594; fold_9...,2.229520,0.254547,1.985944,2.493777,0.507833
7,Tree,LightGBM,Avg_Elapsed_Time,7,WorkloadAware,3,fold_70_10:107.193; fold_80_10:88.1921; fold_9...,103.299420,13.585697,88.192106,...,0.263760,-0.504155,-0.013879,0.490276,fold_70_10:2.53696; fold_80_10:1.83888; fold_9...,2.178438,0.349424,1.838883,2.536959,0.698076
8,Tree,LightGBM,Avg_Elapsed_Time,7,WorkloadPlusActor,3,fold_70_10:89.5601; fold_80_10:106.536; fold_9...,100.004300,9.139322,89.560091,...,0.297360,-0.479527,0.091466,0.570992,fold_70_10:2.09406; fold_80_10:2.21296; fold_9...,2.071216,0.154437,1.906631,2.212960,0.306329
9,Tree,LightGBM,Avg_Remaining_Time,1,PerformanceOnly,3,fold_70_10:61.4718; fold_80_10:91.3074; fold_9...,81.288711,17.162305,61.471797,...,11.086566,-24.097525,-1.935753,22.161771,fold_70_10:0.792048; fold_80_10:1.26427; fold_...,1.222738,0.411499,0.792048,1.611896,0.819848


Rolling-origin results, predictions, models, design matrices, counts, SHAP values, and summaries saved at: /content/direct_prediction_artifacts/rolling_origin_evaluation/tree


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 4. Deep-learning models

Run after Section 1. Section 4.1 defines recurrent helpers and trains LSTM/GRU with and without attention across the three rolling-origin blocks and input variants. Section 4.2 contains the selectable time-series foundation-model blocks.


## 4.1 LSTM and GRU

The helper cell defines the recurrent architectures. The next code cell trains each architecture for all three input variants and saves fold-specific models, preprocessing objects, predictions, sequences, and metrics.


In [ ]:
# RECURRENT MODEL HELPERS FOR ROLLING-ORIGIN EVALUATION
import os,gc,random,logging,joblib
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
from tensorflow.keras.layers import (LSTM,GRU,Dense,Dropout,Input,Bidirectional,Conv1D,
    MaxPooling1D,GlobalAveragePooling1D,LayerNormalization,MultiHeadAttention)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.get_logger().setLevel("ERROR")
TIME_STEPS=15
N_SPLITS=5

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# ============================================================
# Sequence Data
# ============================================================

def create_seq_data(df, features, time_steps=15, target_col="target_scaled"):
    X, y, idx = [], [], []

    for i in range(time_steps - 1, len(df)):
        X.append(df[features].iloc[i - time_steps + 1:i + 1].values)
        y.append(df[target_col].iloc[i])
        idx.append(df.index[i])

    return np.array(X), np.array(y), idx


# ============================================================
# Train / Validation Split
# ============================================================

def make_train_val_split(X_train, y_train, val_fraction=0.15, gap=0):
    """Chronological fit/validation split with a purged gap before validation."""
    val_size = max(int(val_fraction * len(X_train)), 1)
    val_start = len(X_train) - val_size
    fit_end = val_start - gap

    if fit_end <= 0:
        raise ValueError(
            f"Not enough training sequences: n={len(X_train)}, "
            f"val_size={val_size}, gap={gap}"
        )

    X_fit = X_train[:fit_end]
    y_fit = y_train[:fit_end]
    X_val = X_train[val_start:]
    y_val = y_train[val_start:]

    return X_fit, y_fit, X_val, y_val
def build_attention_rnn(input_shape, rnn_type="gru"):
    input_layer = Input(shape=input_shape)

    x = Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        padding="same"
    )(input_layer)

    x = MaxPooling1D(pool_size=2)(x)

    if rnn_type == "lstm":
        x = Bidirectional(LSTM(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = LSTM(32, return_sequences=True)(x)
    else:
        x = Bidirectional(GRU(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = GRU(32, return_sequences=True)(x)

    x = Dropout(0.2)(x)

    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=16
    )(x, x)

    x = LayerNormalization(epsilon=1e-6)(x + attn)

    x = GlobalAveragePooling1D()(x)

    output = Dense(1)(x)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model
def final_evaluate_rnn_for_target_horizon(
    df_train_full,
    df_test_final,
    target_var,
    horizon,
    feature_set="baseline",
    rnn_type="gru",
    time_steps=15,
    seed=42
):
    set_seed(seed)

    CONTEXT_ROWS = 200

    df_train_fe, baseline_features, actor_features = feature_engineering(
        df_train_full,
        target_var=target_var,
        horizon=horizon
    )

    df_context = pd.concat([
        df_train_full.tail(CONTEXT_ROWS),
        df_test_final
    ])

    df_test_fe, _, _ = feature_engineering(
        df_context,
        target_var=target_var,
        horizon=horizon
    )

    df_train_fe.dropna(subset=list(dict.fromkeys(baseline_features + actor_features + ["target"])), inplace=True)
    df_test_fe.dropna(subset=list(dict.fromkeys(baseline_features + actor_features + ["target"])), inplace=True)

    df_train_fe = df_train_fe.copy()
    df_test_fe = df_test_fe.copy()

    if feature_set == "baseline":
        features = baseline_features
    elif feature_set == "workload":
        features = baseline_features + FEATURE_GROUPS["workload"]
    else:
        features = baseline_features + actor_features

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
    df_test_fe[features] = scaler_x.transform(df_test_fe[features])

    df_train_fe["target_scaled"] = scaler_y.fit_transform(
        df_train_fe[["target"]]
    )

    df_test_fe["target_scaled"] = scaler_y.transform(
        df_test_fe[["target"]]
    )

    X_train, y_train, _ = create_seq_data(
        df_train_fe,
        features,
        time_steps,
        "target_scaled"
    )

    X_test, y_test, test_seq_idx = create_seq_data(
        df_test_fe,
        features,
        time_steps,
        "target_scaled"
    )
    test_mask = pd.Index(test_seq_idx).isin(df_test_final.index)
    X_test, y_test = X_test[test_mask], y_test[test_mask]
    test_seq_idx = list(pd.Index(test_seq_idx)[test_mask])

    if (
        X_train.ndim != 3 or
        X_test.ndim != 3 or
        len(X_train) == 0 or
        len(X_test) == 0
    ):
        raise ValueError(
            f"No valid holdout sequences created. "
            f"Target={target_var}, Horizon={horizon}, Feature={feature_set}, "
            f"Train rows={len(df_train_fe)}, Test rows={len(df_test_fe)}, "
            f"X_train={X_train.shape}, X_test={X_test.shape}"
        )

    X_fit, y_fit, X_val, y_val = make_train_val_split(
        X_train,
        y_train,
        val_fraction=0.15,
        gap= horizon
    )

    model = build_attention_rnn(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        rnn_type=rnn_type
    )

    callbacks = [
        EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor="val_loss"
        ),
        ReduceLROnPlateau(
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = model.predict(X_test, verbose=0).flatten()

    y_pred_final = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true_final = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    result = {
        "index": pd.Index(test_seq_idx),
        "y_true": np.asarray(y_true_final),
        "y_pred": np.asarray(y_pred_final),
        "X_test_scaled": np.asarray(X_test),
        "feature_names": list(features),
        "time_steps": int(time_steps),
        "n_train_sequences": int(len(X_train)),
        "n_test_sequences": int(len(X_test)),
        "scaler_x": scaler_x, "scaler_y": scaler_y
    }

    model_dir = ARTIFACT_ROOT / "models" / "deep"
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{rnn_type.upper()}_Attention_{target_var}_h{horizon}_{feature_set}_seed{seed}.keras"
    model.save(model_path)
    joblib.dump({"scaler_x": scaler_x, "scaler_y": scaler_y, "features": features, "target": target_var, "horizon": horizon, "feature_set": feature_set, "target_mode": "direct"}, str(model_path) + ".preprocessing.joblib")
    result["model_path"] = str(model_path)
    tf.keras.backend.clear_session()
    gc.collect()

    return result

final_evaluate_rnn_with_attention=final_evaluate_rnn_for_target_horizon

def build_no_attention_rnn(input_shape, rnn_type="gru"):
    input_layer = Input(shape=input_shape)

    x = Conv1D(filters=64, kernel_size=3, activation="relu", padding="same")(input_layer)
    x = MaxPooling1D(pool_size=2)(x)

    if rnn_type == "lstm":
        x = Bidirectional(LSTM(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = LSTM(32, return_sequences=True)(x)
    else:
        x = Bidirectional(GRU(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = GRU(32, return_sequences=True)(x)

    x = Dropout(0.2)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer="adam", loss="mse")
    return model
def final_evaluate_rnn_for_target_horizon(
    df_train_full,
    df_test_final,
    target_var,
    horizon,
    feature_set="baseline",
    rnn_type="gru",
    time_steps=15,
    seed=42
):
    set_seed(seed)
    df_train_fe, baseline_features, actor_features = feature_engineering(
        df_train_full,
        target_var=target_var,
        horizon=horizon
    )

    df_context = pd.concat([df_train_full.tail(50), df_test_final])
    df_test_fe, _, _ = feature_engineering(
        df_context,
        target_var=target_var,
        horizon=horizon
    )
    df_train_fe.dropna(subset=list(dict.fromkeys(baseline_features + actor_features + ["target"])), inplace=True)
    df_test_fe.dropna(subset=list(dict.fromkeys(baseline_features + actor_features + ["target"])), inplace=True)

    if feature_set == "baseline":
        features = baseline_features
    elif feature_set == "workload":
        features = baseline_features + FEATURE_GROUPS["workload"]
    else:
        features = baseline_features + actor_features

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
    df_test_fe[features] = scaler_x.transform(df_test_fe[features])

    df_train_fe["target_scaled"] = scaler_y.fit_transform(df_train_fe[["target"]])
    df_test_fe["target_scaled"] = scaler_y.transform(df_test_fe[["target"]])

    X_train, y_train, _ = create_seq_data(df_train_fe, features, time_steps, "target_scaled")
    X_test, y_test, test_seq_idx = create_seq_data(df_test_fe, features, time_steps, "target_scaled")
    test_mask = pd.Index(test_seq_idx).isin(df_test_final.index)
    X_test, y_test = X_test[test_mask], y_test[test_mask]
    test_seq_idx = list(pd.Index(test_seq_idx)[test_mask])

    model = build_no_attention_rnn((X_train.shape[1], X_train.shape[2]), rnn_type=rnn_type)

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
    ]

    X_fit, y_fit, X_val, y_val = make_train_val_split(X_train, y_train, val_fraction=0.15, gap= horizon)
    model.fit(
        X_fit, y_fit,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    y_pred_final = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true_final = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    model_dir = ARTIFACT_ROOT / "models" / "deep"
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{rnn_type.upper()}_NoAttention_{target_var}_h{horizon}_{feature_set}.keras"
    model.save(model_path)
    joblib.dump({"scaler_x": scaler_x, "scaler_y": scaler_y, "features": features, "target": target_var, "horizon": horizon, "feature_set": feature_set, "target_mode": "direct"}, str(model_path) + ".preprocessing.joblib")

    return {
        "index": pd.Index(test_seq_idx),
        "y_true": y_true_final,
        "y_pred": y_pred_final,
        "model": model,
        "features": features,
        "X_test": X_test,
        "n_train_sequences": int(len(X_train)),
        "n_test_sequences": int(len(X_test)),
        "scaler_x": scaler_x, "scaler_y": scaler_y
    }


The recurrent rolling-origin evaluator in the next cell trains LSTM and GRU with and without attention for all three variants, saving fold-specific models, preprocessing, predictions, and sequences.


The next cell trains the recurrent models on the three expanding-window blocks and downloads their results bundle.


new

In [ ]:
run_primary_rolling_evaluation("deep")

,group,model,fold,target,horizon_days,variant,n_train,n_test,n_train_raw,n_test_origins_before_horizon_purge,RMSE,MAE,R2,MASE
0,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,99.283315,81.197115,-0.170883,2.415310
1,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,92.246559,72.034966,-0.010791,2.142770
2,DeepLearning,GRU+NoAttention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,97.012207,77.778966,-0.117928,2.313633
3,DeepLearning,GRU+Attention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,93.865197,74.569652,-0.046575,2.218168
4,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,83.507316,65.207919,0.171657,1.939691
5,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,87.290094,69.987556,0.094911,2.081868
6,DeepLearning,GRU+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,86.961066,68.120402,0.101722,2.026327
7,DeepLearning,GRU+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,86.604315,68.196070,0.109077,2.028578
8,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,783,116,821,117,80.427032,62.610605,0.231639,1.862431
9,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,783,116,821,117,87.231142,69.640728,0.096134,2.071551


,group,model,target,horizon_days,variant,n_folds,RMSE_fold_scores,RMSE_mean,RMSE_sd,RMSE_min,...,R2_sd,R2_min,R2_max,R2_range,MASE_fold_scores,MASE_mean,MASE_sd,MASE_min,MASE_max,MASE_range
0,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,PerformanceOnly,3,fold_70_10:93.8652; fold_80_10:96.3116; fold_9...,102.403091,12.728368,93.865197,...,0.098301,-0.240573,-0.046575,0.193998,fold_70_10:2.21817; fold_80_10:2.03329; fold_9...,2.158472,0.108449,2.033291,2.223958,0.190667
1,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,WorkloadAware,3,fold_70_10:86.6043; fold_80_10:90.7742; fold_9...,96.471436,13.639399,86.604315,...,0.106631,-0.102023,0.109077,0.211100,fold_70_10:2.02858; fold_80_10:1.82643; fold_9...,1.968010,0.123040,1.826427,2.049025,0.222599
2,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,WorkloadPlusActor,3,fold_70_10:82.1287; fold_80_10:92.0306; fold_9...,94.294069,13.440805,82.128710,...,0.165775,-0.132740,0.198781,0.331521,fold_70_10:1.88673; fold_80_10:1.87335; fold_9...,1.914730,0.060453,1.873350,1.984106,0.110756
3,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,PerformanceOnly,3,fold_70_10:94.9052; fold_80_10:93.1122; fold_9...,98.634795,8.062693,93.112154,...,0.080916,-0.143738,0.009778,0.153516,fold_70_10:2.26275; fold_80_10:1.95387; fold_9...,2.066482,0.170588,1.953871,2.262748,0.308877
4,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,WorkloadAware,3,fold_70_10:85.636; fold_80_10:88.9803; fold_90...,94.314856,12.250645,85.635971,...,0.071143,-0.044484,0.095151,0.139635,fold_70_10:1.98493; fold_80_10:1.82826; fold_9...,1.927635,0.086399,1.828258,1.984935,0.156677
5,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,WorkloadPlusActor,3,fold_70_10:84.1017; fold_80_10:100.912; fold_9...,96.685580,11.091891,84.101749,...,0.254839,-0.343387,0.127283,0.470670,fold_70_10:2.00658; fold_80_10:2.04656; fold_9...,1.981388,0.080770,1.891023,2.046558,0.155535
6,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,PerformanceOnly,3,fold_70_10:101.393; fold_80_10:99.7494; fold_9...,106.696374,10.641269,99.749413,...,0.079482,-0.345779,-0.190368,0.155411,fold_70_10:2.40736; fold_80_10:2.08372; fold_9...,2.259491,0.163615,2.083716,2.407356,0.323640
7,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,WorkloadAware,3,fold_70_10:89.1196; fold_80_10:92.7341; fold_9...,98.549797,13.326417,89.119590,...,0.040992,-0.121001,-0.039700,0.081301,fold_70_10:2.11438; fold_80_10:1.83416; fold_9...,2.030542,0.170676,1.834160,2.143086,0.308926
8,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,WorkloadPlusActor,3,fold_70_10:96.1817; fold_80_10:130.787; fold_9...,109.758094,18.466936,96.181660,...,0.703229,-1.229738,0.119408,1.349146,fold_70_10:2.37529; fold_80_10:2.80501; fold_9...,2.337483,0.487537,1.832142,2.805014,0.972873
9,DeepLearning,GRU+Attention,Avg_Remaining_Time,1,PerformanceOnly,3,fold_70_10:93.4109; fold_80_10:89.8626; fold_9...,83.604058,14.025679,67.538686,...,3.790915,-12.656589,-5.778953,6.877635,fold_70_10:1.36875; fold_80_10:1.51326; fold_9...,1.338489,0.191705,1.133452,1.513262,0.379810


Rolling-origin results, predictions, models, design matrices, counts, SHAP values, and summaries saved at: /content/direct_prediction_artifacts/rolling_origin_evaluation/deep


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

old

In [ ]:
run_primary_rolling_evaluation("deep")


,group,model,fold,target,horizon_days,variant,n_train,n_test,n_train_raw,n_test_origins_before_horizon_purge,RMSE,MAE,R2,MASE
0,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,95.410934,74.714593,-0.081328,2.222479
1,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,91.376369,73.076220,0.008189,2.173744
2,DeepLearning,GRU+NoAttention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,95.324097,75.125121,-0.079361,2.234691
3,DeepLearning,GRU+Attention,fold_70_10,Avg_Elapsed_Time,1,PerformanceOnly,783,116,821,117,105.593512,89.210353,-0.324450,2.653674
4,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,85.822405,67.644289,0.125092,2.012164
5,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,86.913982,67.084670,0.102694,1.995518
6,DeepLearning,GRU+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,83.365660,64.402238,0.174465,1.915725
7,DeepLearning,GRU+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadAware,783,116,821,117,87.951520,68.297985,0.081143,2.031609
8,DeepLearning,LSTM+NoAttention,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,783,116,821,117,81.219123,63.943359,0.216430,1.902075
9,DeepLearning,LSTM+Attention,fold_70_10,Avg_Elapsed_Time,1,WorkloadPlusActor,783,116,821,117,97.361433,79.025933,-0.125991,2.350726


,group,model,target,horizon_days,variant,n_folds,RMSE_fold_scores,RMSE_mean,RMSE_sd,RMSE_min,...,R2_sd,R2_min,R2_max,R2_range,MASE_fold_scores,MASE_mean,MASE_sd,MASE_min,MASE_max,MASE_range
0,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,PerformanceOnly,3,fold_70_10:105.594; fold_80_10:90.736; fold_90...,103.042619,11.250216,90.735985,...,0.151027,-0.324450,-0.036672,0.287778,fold_70_10:2.65367; fold_80_10:1.90758; fold_9...,2.223399,0.385991,1.907580,2.653674,0.746094
1,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,WorkloadAware,3,fold_70_10:87.9515; fold_80_10:91.3408; fold_9...,96.801018,12.507879,87.951520,...,0.098705,-0.115824,0.081143,0.196967,fold_70_10:2.03161; fold_80_10:1.83801; fold_9...,1.968450,0.112987,1.838005,2.035735,0.197730
2,DeepLearning,GRU+Attention,Avg_Elapsed_Time,1,WorkloadPlusActor,3,fold_70_10:81.341; fold_80_10:94.5712; fold_90...,93.595738,11.797306,81.340972,...,0.212301,-0.196145,0.214077,0.410222,fold_70_10:1.86156; fold_80_10:1.92102; fold_9...,1.892106,0.029760,1.861563,1.921017,0.059453
3,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,PerformanceOnly,3,fold_70_10:94.9645; fold_80_10:94.4908; fold_9...,99.200488,7.750761,94.490805,...,0.092690,-0.177858,0.005017,0.182875,fold_70_10:2.28394; fold_80_10:1.9553; fold_90...,2.076317,0.180626,1.955297,2.283937,0.328641
4,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,WorkloadAware,3,fold_70_10:93.3529; fold_80_10:92.2052; fold_9...,97.743549,8.617941,92.205190,...,0.068752,-0.121566,0.013711,0.135276,fold_70_10:2.19005; fold_80_10:1.84895; fold_9...,1.994440,0.175989,1.848950,2.190052,0.341102
5,DeepLearning,GRU+Attention,Avg_Elapsed_Time,3,WorkloadPlusActor,3,fold_70_10:91.5081; fold_80_10:103.253; fold_9...,101.131740,8.757908,91.508129,...,0.224392,-0.406426,-0.003987,0.402439,fold_70_10:2.23114; fold_80_10:2.06913; fold_9...,2.079136,0.147256,1.937138,2.231140,0.294002
6,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,PerformanceOnly,3,fold_70_10:100.727; fold_80_10:93.5986; fold_9...,100.340381,6.556936,93.598579,...,0.185201,-0.328176,0.042224,0.370400,fold_70_10:2.36422; fold_80_10:1.93177; fold_9...,2.087830,0.240029,1.931766,2.364223,0.432457
7,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,WorkloadAware,3,fold_70_10:90.0806; fold_80_10:88.1748; fold_9...,95.415481,10.932364,88.174839,...,0.040809,-0.062243,0.018820,0.081063,fold_70_10:2.1961; fold_80_10:1.82271; fold_90...,1.999367,0.187502,1.822712,2.196099,0.373387
8,DeepLearning,GRU+Attention,Avg_Elapsed_Time,7,WorkloadPlusActor,3,fold_70_10:85.5849; fold_80_10:93.3132; fold_9...,94.512973,9.584428,85.584894,...,0.114139,-0.135046,0.078755,0.213801,fold_70_10:2.0529; fold_80_10:1.82432; fold_90...,1.918637,0.119410,1.824316,2.052898,0.228581
9,DeepLearning,GRU+Attention,Avg_Remaining_Time,1,PerformanceOnly,3,fold_70_10:123.314; fold_80_10:133.38; fold_90...,110.501060,31.317717,74.808488,...,8.625497,-27.598318,-10.813973,16.784345,fold_70_10:1.83581; fold_80_10:2.26649; fold_9...,1.806401,0.475476,1.316901,2.266487,0.949586


Rolling-origin results, predictions, models, design matrices, counts, SHAP values, and summaries saved at: /content/direct_prediction_artifacts/rolling_origin_evaluation/deep


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4.2 Time-series foundation models

The pretrained backbones remain frozen; the BPIC-trained prediction layer maps their forecasts of permitted input series to the direct KPI targets. Run the setup cell once, then run any selected model block.


In [ ]:
#@title 4.2.0 Foundation-model setup (run once before one or more model blocks)
# Pretrained model weights remain frozen. Their forecasts of allowed inputs feed
# a BPIC-trained Ridge prediction layer that directly predicts future KPI levels.
import os, re, json, time, gc, importlib.metadata, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

# Install only the selected model package in a separate Colab cell (commands are listed above).
# Restart Colab after changing core packages, then rerun the setup cell.
FM_HORIZONS=[1,3,7]; FM_MAX_HORIZON=max(FM_HORIZONS); FM_CONTEXT_LENGTH=512
FM_MIN_HISTORY=32; FM_BATCH_SIZE=64; FM_RIDGE_ALPHA=1.0
FM_DEVICE="cuda" if __import__("torch").cuda.is_available() else "cpu"
FM_TARGETS=["Avg_Elapsed_Time","Avg_Remaining_Time"]

# Agreed predictor variants; Avg_Remaining_Time is never included as an input.
FM_BASE_SERIES={"Avg_Elapsed_Time":pd.to_numeric(df["Avg_Elapsed_Time"],errors="coerce")}
for _col in PROCESS_STATE_VARS:
    if _col not in df.columns: raise ValueError(f"Required process-state input is missing: {_col}")
    FM_BASE_SERIES[_col]=pd.to_numeric(df[_col],errors="coerce")
FM_COUNT_COLS=sorted(c for c in df.columns if c.startswith("Cum_Count_"))
FM_DURATION_COLS=sorted(c for c in df.columns if c.startswith("Cum_Time_") and c.endswith("_seconds"))
FM_COUNT_FRAME=df[FM_COUNT_COLS].apply(pd.to_numeric,errors="coerce") if FM_COUNT_COLS else pd.DataFrame(index=df.index)
FM_TOTAL_COUNTS=FM_COUNT_FRAME.sum(axis=1) if FM_COUNT_COLS else pd.Series(0.0,index=df.index)
FM_PROPORTION_COLS=[]
for _count_col in FM_COUNT_COLS:
    _class=_count_col[len("Cum_Count_"):]; _prop_col=f"Behavior_Proportion_{_class}"
    FM_BASE_SERIES[_prop_col]=FM_COUNT_FRAME[_count_col].div(FM_TOTAL_COUNTS.replace(0,np.nan)).fillna(0.0)
    FM_PROPORTION_COLS.append(_prop_col)
for _col in FM_COUNT_COLS+FM_DURATION_COLS: FM_BASE_SERIES[_col]=pd.to_numeric(df[_col],errors="coerce")
FM_VARIANTS={
 "PerformanceOnly":["Avg_Elapsed_Time"],
 "WorkloadAware":["Avg_Elapsed_Time"]+list(PROCESS_STATE_VARS),
 "WorkloadPlusActor":["Avg_Elapsed_Time"]+list(PROCESS_STATE_VARS)+FM_PROPORTION_COLS+FM_COUNT_COLS+FM_DURATION_COLS}
FM_VARIANTS={k:list(dict.fromkeys(v)) for k,v in FM_VARIANTS.items()}
for _variant,_cols in FM_VARIANTS.items():
    _missing=[c for c in _cols if c not in FM_BASE_SERIES]
    if _missing: raise ValueError(f"Missing {_variant} inputs: {_missing}")
# Causal forward fill only; no future observations are used.
for _col in FM_BASE_SERIES: FM_BASE_SERIES[_col]=FM_BASE_SERIES[_col].replace([np.inf,-np.inf],np.nan).ffill().fillna(0.0)
FM_SPLIT_POS=len(df_train_full)
FM_FOLD_SPECS=[("fold_70_10",0.70,0.80),("fold_80_10",0.80,0.90),("fold_90_10",0.90,1.00)]
FM_ORIGIN_POSITIONS=list(range(FM_MIN_HISTORY-1,max(FM_MIN_HISTORY-1,len(df)-FM_MAX_HORIZON)))
if not FM_ORIGIN_POSITIONS: raise ValueError("Not enough rows for the selected history and horizon settings.")
FM_ALL_INPUTS=list(dict.fromkeys(c for cols in FM_VARIANTS.values() for c in cols))
FM_DATASET_TAG=re.sub(r"[^A-Za-z0-9_.-]+","_",Path(csv_path).stem)
FM_ARTIFACT_ROOT=Path(ARTIFACT_ROOT)/"foundation_model_results"/FM_DATASET_TAG; FM_ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)

def _fm_predict_batch(model_name,model_object,contexts):
    """Return point forecasts for a batch of univariate histories."""
    if model_name=="TimesFM-2.5":
        point,_quantiles=model_object.forecast(horizon=FM_MAX_HORIZON,inputs=contexts)
        return np.asarray(point,dtype=float).reshape(len(contexts),-1)[:,:FM_MAX_HORIZON]
    if model_name=="Chronos-2":
        import torch
        outputs=model_object.predict(inputs=[torch.as_tensor(x,dtype=torch.float32) for x in contexts],
             prediction_length=FM_MAX_HORIZON,batch_size=min(FM_BATCH_SIZE,len(contexts)),cross_learning=False)
        rows=[]
        for out in outputs:
            arr=out.detach().cpu().numpy() if hasattr(out,"detach") else np.asarray(out); arr=np.squeeze(arr)
            if arr.ndim==2: arr=arr[arr.shape[0]//2,:]
            elif arr.ndim>2:
                arr=arr.reshape(-1,arr.shape[-2],arr.shape[-1])[0]; arr=arr[arr.shape[0]//2,:]
            rows.append(np.asarray(arr,dtype=float).reshape(-1)[:FM_MAX_HORIZON])
        return np.vstack(rows)
    if model_name=="MOIRAI-2.0":
        predictor,ListDataset=model_object
        entries=[{"start":pd.Period("2000-01-01",freq="D"),"target":np.asarray(x,dtype=np.float32)} for x in contexts]
        forecasts=list(predictor.predict(ListDataset(entries,freq="D")))
        return np.vstack([np.asarray(f.quantile(0.5),dtype=float).reshape(-1)[:FM_MAX_HORIZON] for f in forecasts])
    raise KeyError(model_name)

def _fm_mase(y_true, y_pred, train_values):
    y_true=np.asarray(y_true,dtype=float); y_pred=np.asarray(y_pred,dtype=float)
    scale=float(np.mean(np.abs(np.diff(np.asarray(train_values,dtype=float))))) if len(train_values)>1 else np.nan
    return float(mean_absolute_error(y_true,y_pred)/scale) if np.isfinite(scale) and scale>0 else np.nan

def run_frozen_foundation_model(model_name,model_object,checkpoint):
    """Frozen-model input forecasts + fold-specific supervised direct-target heads."""
    out=FM_ARTIFACT_ROOT/model_name.replace("-","_")
    for folder in ["models","metrics","predictions","design_matrices","feature_forecasts"]: (out/folder).mkdir(parents=True,exist_ok=True)
    requests=[(col,origin) for col in FM_ALL_INPUTS for origin in FM_ORIGIN_POSITIONS]; cache={}; raw_rows=[]
    started=time.time(); print(f"{model_name}: frozen inference for {len(requests):,} input-series/cutoff pairs")
    for start in tqdm(range(0,len(requests),FM_BATCH_SIZE),desc=f"{model_name} inference"):
        batch=requests[start:start+FM_BATCH_SIZE]; contexts=[]
        for feature,origin in batch:
            hist=FM_BASE_SERIES[feature].iloc[max(0,origin-FM_CONTEXT_LENGTH+1):origin+1].to_numpy(dtype=np.float32)
            contexts.append(hist)
        forecasts=_fm_predict_batch(model_name,model_object,contexts)
        if forecasts.shape[1]<FM_MAX_HORIZON: raise ValueError(f"{model_name} returned fewer than {FM_MAX_HORIZON} steps")
        for (feature,origin),values in zip(batch,forecasts):
            cache[(feature,origin)]=np.asarray(values[:FM_MAX_HORIZON],dtype=float)
            for step,value in enumerate(values[:FM_MAX_HORIZON],start=1):
                raw_rows.append({"input_series":feature,"cutoff_date":df.index[origin],"forecast_step":step,
                    "forecast_date":df.index[min(origin+step,len(df)-1)],"forecast_value":float(value)})
    print(f"{model_name}: inference finished in {(time.time()-started)/60:.1f} minutes")
    pd.DataFrame(raw_rows).to_csv(out/"feature_forecasts"/"frozen_model_input_forecasts.csv.gz",index=False,compression="gzip")
    metric_rows=[]; coefficient_rows=[]; prediction_records=[]
    n=len(df)
    for fold_name,train_fraction,test_end_fraction in FM_FOLD_SPECS:
        train_end=int(np.floor(n*train_fraction)); test_end=int(np.floor(n*test_end_fraction))
        for target in FM_TARGETS:
            for horizon in FM_HORIZONS:
                train_origins=[i for i in FM_ORIGIN_POSITIONS if i+horizon<train_end]
                # Purge the last h origins from the test window: their target
                # would otherwise fall beyond this fold's test boundary.
                test_origins=[i for i in FM_ORIGIN_POSITIONS if i>=train_end and i+horizon<test_end]
                train_values=pd.to_numeric(df[target].iloc[:train_end],errors="coerce").dropna().to_numpy(dtype=float)
                for variant,input_cols in FM_VARIANTS.items():
                    names=[f"FrozenForecast__{c}__t_plus_{horizon}" for c in input_cols]
                    def make_xy(origins):
                        X=np.vstack([[cache[(c,i)][horizon-1] for c in input_cols] for i in origins])
                        y=np.asarray([df[target].iloc[i+horizon] for i in origins],dtype=float)
                        return pd.DataFrame(X,columns=names,index=[df.index[i] for i in origins]),y,origins
                    if len(train_origins)<10 or not test_origins: continue
                    Xtr,ytr,_=make_xy(train_origins); Xte,yte,origte=make_xy(test_origins)
                    tr_ok=np.isfinite(ytr); te_ok=np.isfinite(yte)
                    Xtr,ytr=Xtr.loc[tr_ok],ytr[tr_ok]; Xte,yte=Xte.loc[te_ok],yte[te_ok]
                    origte=[i for i,ok in zip(origte,te_ok) if ok]
                    if len(Xtr)<10 or len(Xte)==0: continue
                    head=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),
                                   ("prediction_layer",Ridge(alpha=FM_RIDGE_ALPHA))])
                    head.fit(Xtr,ytr); pred=np.asarray(head.predict(Xte),dtype=float)
                    scores={"RMSE":float(np.sqrt(mean_squared_error(yte,pred))),"MAE":float(mean_absolute_error(yte,pred)),
                            "R2":float(r2_score(yte,pred)) if len(yte)>1 else np.nan,
                            "MASE":_fm_mase(yte,pred,train_values)}
                    safe=f"{model_name}_{fold_name}_{target}_h{horizon}_{variant}".replace("-","_")
                    metric_rows.append({"group":"FrozenFoundationModel_SupervisedReadout","model":model_name,"fold":fold_name,
                        "train_fraction":train_fraction,"test_window_fraction":0.10,"target":target,"horizon_days":horizon,
                        "variant":variant,"n_train":len(Xtr),"n_test":len(Xte),"n_train_origins_before_feature_filter":len(train_origins),
                        "n_test_origins_before_feature_filter":len(test_origins),"target_mode":"direct_future_level_y_t_plus_h",
                        "readout":"Ridge prediction layer",**scores})
                    joblib.dump({"foundation_model":model_name,"checkpoint":checkpoint,"readout":head,"feature_names":names,
                        "target":target,"horizon_days":horizon,"variant":variant,"fold":fold_name,"ridge_alpha":FM_RIDGE_ALPHA,
                        "target_mode":"direct_future_level_y_t_plus_h","Avg_Remaining_Time_used_as_input":False,
                        "foundation_weights":"frozen; inference only","train_cutoff_dates":list(Xtr.index)},out/"models"/f"{safe}.joblib")
                    Xtr.assign(target=ytr).to_csv(out/"design_matrices"/f"{safe}_train.csv",index_label="cutoff_date")
                    Xte.assign(target=yte,prediction=pred).to_csv(out/"design_matrices"/f"{safe}_test.csv",index_label="cutoff_date")
                    target_dates=[df.index[i+horizon] for i in origte]
                    pred_df=pd.DataFrame({"fold":fold_name,"cutoff_date":Xte.index,"target_date":target_dates,
                        "y_true":yte,"y_pred":pred,"error":yte-pred})
                    pred_df.to_csv(out/"predictions"/f"{safe}.csv",index=False); prediction_records.append(pred_df)
                    for feature,coef in zip(names,head.named_steps["prediction_layer"].coef_):
                        coefficient_rows.append({"model":model_name,"fold":fold_name,"target":target,"horizon_days":horizon,
                            "variant":variant,"feature":feature,"standardized_coefficient":float(coef)})
    fold_metrics=pd.DataFrame(metric_rows)
    fold_metrics.to_csv(out/"metrics"/"rolling_origin_fold_metrics.csv",index=False)
    if len(fold_metrics):
        summary=(fold_metrics.groupby(["group","model","target","horizon_days","variant"],as_index=False)
            .agg({m:["mean","std","min","max"] for m in ["RMSE","MAE","R2","MASE"]}))
        summary.columns=["_".join(c).strip("_") if isinstance(c,tuple) else c for c in summary.columns]
        summary.to_csv(out/"metrics"/"rolling_origin_summary.csv",index=False)
    pd.DataFrame(coefficient_rows).to_csv(out/"metrics"/"prediction_layer_coefficients.csv",index=False)
    config={"model":model_name,"checkpoint":checkpoint,"method":"frozen input-series forecasts plus fold-specific training-only Ridge readout",
        "weights_frozen":True,"targets":FM_TARGETS,"remaining_time_as_input":False,"horizons_days":FM_HORIZONS,
        "variants":FM_VARIANTS,"folds":FM_FOLD_SPECS,"readout":"Ridge","ridge_alpha":FM_RIDGE_ALPHA,
        "context_length":FM_CONTEXT_LENGTH,"input_csv":csv_path,"device":FM_DEVICE}
    try: config["package_versions"]={k:importlib.metadata.version(k) for k in ["timesfm","chronos-forecasting","uni2ts","scikit-learn"]}
    except Exception: pass
    (out/"run_config.json").write_text(json.dumps(config,indent=2,default=str))
    meta_out=out/"dataset_metadata"; meta_out.mkdir(exist_ok=True)
    for _name in ["dataset_characteristics.csv","behavior_type_distribution.csv","input_daily_timeseries.csv","chronological_split.csv"]:
        _src=Path(ARTIFACT_ROOT)/_name
        if _src.exists(): shutil.copy2(_src,meta_out/_name)
    archive=shutil.make_archive(str(Path(ARTIFACT_ROOT)/f"{FM_DATASET_TAG}_{model_name.replace('-','_')}_rolling_origin_results"),"zip",root_dir=out)
    print(f"Saved rolling-origin folds, models, readouts, design matrices, predictions, and metrics to {out}")
    from google.colab import files
    files.download(archive)
    del cache,raw_rows; gc.collect()


In [ ]:
%pip install -q "timesfm[torch]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 11.0 MB/s eta 0:00:00


In [ ]:
#@title 4.2.1 TimesFM-2.5: frozen backbone + BPIC prediction layer
import timesfm
FM_MODEL_NAME="TimesFM-2.5"; FM_CHECKPOINT="google/timesfm-2.5-200m-pytorch"
_fm_model=timesfm.TimesFM_2p5_200M_torch.from_pretrained(FM_CHECKPOINT)
_fm_model.compile(timesfm.ForecastConfig(max_context=FM_CONTEXT_LENGTH,max_horizon=FM_MAX_HORIZON,normalize_inputs=True,use_continuous_quantile_head=True))
run_frozen_foundation_model(FM_MODEL_NAME,_fm_model,FM_CHECKPOINT)
del _fm_model; gc.collect()


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

TimesFM-2.5: frozen inference for 19,295 input-series/cutoff pairs


TimesFM-2.5 inference:   0%|          | 0/302 [00:00<?, ?it/s]

TimesFM-2.5: inference finished in 35.6 minutes
Saved rolling-origin folds, models, readouts, design matrices, predictions, and metrics to /content/direct_prediction_artifacts/foundation_model_results/BPIC2011_final_online_daily_time_series_cutoff_safe_days/TimesFM_2.5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4955

In [ ]:
%pip install -q chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 7.7 MB/s eta 0:00:00


In [ ]:
#@title 4.2.3 Chronos-2: frozen backbone + BPIC prediction layer
from chronos import Chronos2Pipeline
FM_MODEL_NAME="Chronos-2"; FM_CHECKPOINT="amazon/chronos-2"
_fm_model=Chronos2Pipeline.from_pretrained(FM_CHECKPOINT,device_map=FM_DEVICE)
run_frozen_foundation_model(FM_MODEL_NAME,_fm_model,FM_CHECKPOINT)
del _fm_model; gc.collect()

fm_out = FM_ARTIFACT_ROOT / FM_MODEL_NAME.replace("-", "_")

fold_metrics = pd.read_csv(fm_out / "metrics" / "rolling_origin_fold_metrics.csv")
fm_summary = pd.read_csv(fm_out / "metrics" / "rolling_origin_summary.csv")

display(fold_metrics.head(20))
display(fm_summary.head(20))


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2: frozen inference for 19,295 input-series/cutoff pairs


Chronos-2 inference:   0%|          | 0/302 [00:00<?, ?it/s]

Chronos-2: inference finished in 0.9 minutes
Saved rolling-origin folds, models, readouts, design matrices, predictions, and metrics to /content/direct_prediction_artifacts/foundation_model_results/BPIC2011_final_online_daily_time_series_cutoff_safe_days/Chronos_2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,group,model,fold,train_fraction,test_window_fraction,target,horizon_days,variant,n_train,n_test,n_train_origins_before_feature_filter,n_test_origins_before_feature_filter,target_mode,readout,RMSE,MAE,R2,MASE
0,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,1,PerformanceOnly,789,116,789,116,direct_future_level_y_t_plus_h,Ridge prediction layer,73.727960,56.289179,0.354308,1.674392
1,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,1,WorkloadAware,789,116,789,116,direct_future_level_y_t_plus_h,Ridge prediction layer,74.114369,56.278612,0.347522,1.674078
2,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,1,WorkloadPlusActor,789,116,789,116,direct_future_level_y_t_plus_h,Ridge prediction layer,73.783386,56.483037,0.353337,1.680159
3,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,3,PerformanceOnly,787,114,787,114,direct_future_level_y_t_plus_h,Ridge prediction layer,88.221287,67.050702,0.039693,1.994507
4,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,3,WorkloadAware,787,114,787,114,direct_future_level_y_t_plus_h,Ridge prediction layer,88.234377,65.994302,0.039408,1.963083
5,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,3,WorkloadPlusActor,787,114,787,114,direct_future_level_y_t_plus_h,Ridge prediction layer,85.501302,65.173675,0.097995,1.938673
6,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,7,PerformanceOnly,783,110,783,110,direct_future_level_y_t_plus_h,Ridge prediction layer,88.130613,69.542406,-0.016753,2.068626
7,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,7,WorkloadAware,783,110,783,110,direct_future_level_y_t_plus_h,Ridge prediction layer,87.420751,68.210935,-0.000440,2.029020
8,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Elapsed_Time,7,WorkloadPlusActor,783,110,783,110,direct_future_level_y_t_plus_h,Ridge prediction layer,84.365191,65.938587,0.068274,1.961426
9,FrozenFoundationModel_SupervisedReadout,Chronos-2,fold_70_10,0.7,0.1,Avg_Remaining_Time,1,PerformanceOnly,789,116,789,116,direct_future_level_y_t_plus_h,Ridge prediction layer,112.745659,84.107984,-8.875689,1.341145


,group,model,target,horizon_days,variant,RMSE_mean,RMSE_std,RMSE_min,RMSE_max,MAE_mean,...,MAE_min,MAE_max,R2_mean,R2_std,R2_min,R2_max,MASE_mean,MASE_std,MASE_min,MASE_max
0,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,1,PerformanceOnly,81.687464,11.454540,73.727960,94.815506,63.029069,...,56.289179,74.293960,0.255524,0.086228,0.195339,0.354308,1.662751,0.107831,1.549571,1.764289
1,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,1,WorkloadAware,81.748162,11.488597,74.114369,94.960896,63.116772,...,56.278612,74.614519,0.254819,0.081784,0.192869,0.347522,1.664770,0.112076,1.548329,1.771901
2,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,1,WorkloadPlusActor,80.756908,11.155249,73.783386,93.622746,62.581470,...,56.483037,73.491607,0.273071,0.071677,0.215456,0.353337,1.651839,0.110317,1.530122,1.745235
3,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,3,PerformanceOnly,94.400053,7.197581,88.221287,102.303199,73.003278,...,67.050702,80.696112,-0.005630,0.111861,-0.133040,0.076457,1.932781,0.055364,1.887512,1.994507
4,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,3,WorkloadAware,94.356987,6.799349,88.234377,101.674626,72.617650,...,65.994302,80.315911,-0.005928,0.122812,-0.144962,0.087771,1.921766,0.036313,1.894921,1.963083
5,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,3,WorkloadPlusActor,90.566529,8.383285,85.501302,100.243166,69.715842,...,65.173675,77.278337,0.078867,0.046990,0.025330,0.113277,1.846789,0.086656,1.766534,1.938673
6,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,7,PerformanceOnly,94.834050,6.372241,88.130613,100.813202,73.614629,...,69.542406,77.615674,-0.017392,0.172610,-0.190321,0.154897,1.954493,0.112754,1.843171,2.068626
7,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,7,WorkloadAware,94.370139,6.428027,87.420751,100.102939,73.089579,...,68.210935,77.340647,-0.008235,0.179023,-0.191028,0.166763,1.939391,0.096859,1.836640,2.029020
8,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Elapsed_Time,7,WorkloadPlusActor,90.918534,8.156916,84.365191,100.054007,70.133033,...,65.938587,76.586045,0.072884,0.092475,-0.017200,0.167578,1.859302,0.089060,1.797761,1.961426
9,FrozenFoundationModel_SupervisedReadout,Chronos-2,Avg_Remaining_Time,1,PerformanceOnly,119.887950,16.866727,107.767466,139.150725,95.855157,...,84.107984,109.767973,-29.780020,28.927794,-62.794853,-8.875689,1.678556,0.374770,1.341145,2.081927
